# TIC TAC TOE 

Política → Estrategia utilizada por el agente para seleccionar la mejor acción posible. En el juego Tic-Tac-Toe 4×4, la política es de tipo greedy, ya que el agente evalúa todos los movimientos válidos y elige aquel que conduce al estado con mayor valor estimado por la función de valor.

Estado → Representación actual del tablero de juego. El estado está definido por una matriz 4×4 donde:
1 representa las fichas del agente (X),
-1 representa las fichas del jugador humano (O),
0 representa casillas vacías.

Acción → Movimiento realizado por un jugador sobre el tablero. Una acción consiste en colocar una ficha en una posición válida (fila,columna) disponible dentro del tablero.
Recompensa → Valor numérico que indica el resultado de una partida para el agente:
+1 si el agente gana,
−1 si el agente pierde,
0 si ocurre un empate.

Función de valor → Función que estima qué tan favorable es un estado del tablero para el agente. Cada estado tiene asociado un valor numérico almacenado en value_function, y el agente utiliza esos valores para decidir cuál es el mejor movimiento posible.

In [1]:
import numpy as np
import pickle

In [2]:
with open('agente2do.pickle', 'rb') as handle:
    funcion_de_valor = pickle.load(handle)

len(funcion_de_valor)

1250461

In [3]:
import numpy as np

class Board:
    def __init__(self):
        self.state = np.zeros((4, 4))

    def valid_moves(self):
        return [(i, j) for i in range(4) for j in range(4) if self.state[i, j] == 0]

    def update(self, symbol, row, col):
        if self.state[row, col] != 0:
            raise ValueError('movimiento ilegal')
        self.state[row, col] = symbol

    def is_game_over(self):
        # Filas y columnas
        if (self.state.sum(axis=1) == 4).any() or (self.state.sum(axis=0) == 4).any():
            return 1
        if (self.state.sum(axis=1) == -4).any() or (self.state.sum(axis=0) == -4).any():
            return -1

        # Diagonales
        diag_sums = [
            sum(self.state[i, i]     for i in range(4)),
            sum(self.state[i, 3 - i] for i in range(4)),
        ]
        if diag_sums[0] == 4 or diag_sums[1] == 4:
            return 1
        if diag_sums[0] == -4 or diag_sums[1] == -4:
            return -1

        # Empate
        if len(self.valid_moves()) == 0:
            return 0

        return None  # partida en curso


class AgentInferencia:
    def __init__(self, value_function, symbol=1):
        self.value_function = value_function
        self.symbol = symbol

    def move(self, board):
        valid_moves = board.valid_moves()
        max_value = -1000
        best_row, best_col = valid_moves[0]

        for row, col in valid_moves:
            next_board = board.state.copy()
            next_board[row, col] = self.symbol
            next_state = str(next_board.reshape(16))
            value = self.value_function.get(next_state, 0)

            if value >= max_value:
                max_value = value
                best_row, best_col = row, col

        return best_row, best_col

In [4]:
def dibujar_tablero(board):
    simbolos = {1: 'X', -1: 'O', 0: ' '}
    print('')
    for r in range(4):
        fila = []
        for c in range(4):
            if board.state[r, c] == 0:
                fila.append(str(r * 4 + c + 1))
            else:
                fila.append(simbolos[int(board.state[r, c])])
        print(' ' + ' | '.join(fila))
        if r < 3:
            print('----+----+----+----')
    print('')


def movimiento_humano(board):
    valid_numbers = {r * 4 + c + 1: (r, c) for r, c in board.valid_moves()}
    while True:
        entrada = input(f'Tu turno (elige {sorted(valid_numbers.keys())}): ').strip()
        if not entrada.isdigit():
            print('Entrada invalida. Escribe un numero.')
            continue
        numero = int(entrada)
        if numero not in valid_numbers:
            print('Esa casilla no esta disponible.')
            continue
        return valid_numbers[numero]


def jugar(agente_empieza=True):
    board = Board()
    agente = AgentInferencia(funcion_de_valor, symbol=1)
    simbolo_humano = -1

    print('Humano: O | Agente: X')
    dibujar_tablero(board)

    turno_agente = agente_empieza
    while board.is_game_over() is None:
        if turno_agente:
            row, col = agente.move(board)
            board.update(agente.symbol, row, col)
            print(f'Agente juega en casilla {row * 4 + col + 1}')
        else:
            row, col = movimiento_humano(board)
            board.update(simbolo_humano, row, col)

        dibujar_tablero(board)
        turno_agente = not turno_agente

    resultado = board.is_game_over()
    if resultado == 1:
        print('Gana el agente.')
    elif resultado == -1:
        print('Ganaste.')
    else:
        print('Empate.')

In [ ]:
# Ejecuta esta celda para jugar una partida.
# Cambia a False si quieres empezar tu.
jugar(agente_empieza=False)

Humano: O | Agente: X

 1 | 2 | 3 | 4
----+----+----+----
 5 | 6 | 7 | 8
----+----+----+----
 9 | 10 | 11 | 12
----+----+----+----
 13 | 14 | 15 | 16


 O | 2 | 3 | 4
----+----+----+----
 5 | 6 | 7 | 8
----+----+----+----
 9 | 10 | 11 | 12
----+----+----+----
 13 | 14 | 15 | 16

Agente juega en casilla 10

 O | 2 | 3 | 4
----+----+----+----
 5 | 6 | 7 | 8
----+----+----+----
 9 | X | 11 | 12
----+----+----+----
 13 | 14 | 15 | 16


 O | O | 3 | 4
----+----+----+----
 5 | 6 | 7 | 8
----+----+----+----
 9 | X | 11 | 12
----+----+----+----
 13 | 14 | 15 | 16

Agente juega en casilla 4

 O | O | 3 | X
----+----+----+----
 5 | 6 | 7 | 8
----+----+----+----
 9 | X | 11 | 12
----+----+----+----
 13 | 14 | 15 | 16

Esa casilla no esta disponible.

 O | O | 3 | X
----+----+----+----
 O | 6 | 7 | 8
----+----+----+----
 9 | X | 11 | 12
----+----+----+----
 13 | 14 | 15 | 16

Agente juega en casilla 16

 O | O | 3 | X
----+----+----+----
 O | 6 | 7 | 8
----+----+----+----
 9 | X | 11 | 12
----+---